# Fine-tune PPE Model for Gloves Detection

Fine-tunes the existing `ppe_model.pt` on a proper gloves dataset with **real bounding box labels** from Roboflow.

**Before running:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Upload your `ppe_model.pt` in Cell 2
3. Run all cells top to bottom
4. Download `ppe_gloves.pt` at the end and place it in `weights/`

In [ ]:
# Cell 1 — Install dependencies
!pip install -q ultralytics roboflow

In [ ]:
# Cell 2 — Upload ppe_model.pt from your computer
from google.colab import files
uploaded = files.upload()   # select weights/ppe_model.pt from your project

import os, shutil
shutil.copy('ppe_model.pt', '/content/ppe_model.pt')
print('Base model ready:', os.path.exists('/content/ppe_model.pt'))

In [ ]:
# Cell 3 — Download gloves dataset from Roboflow (has real bounding box labels)
# Dataset: Safety Gloves Detection — 2 classes: Gloves, NO-Gloves
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")   # ← get free key at roboflow.com
project = rf.workspace("roboflow-universe-projects").project("safety-gloves-detection")
dataset = project.version(1).download("yolov8")

DATA_YAML = dataset.location + "/data.yaml"
print("Dataset at:", dataset.location)

import yaml
with open(DATA_YAML) as f:
    print(f.read())

In [ ]:
# Cell 4 — Inspect class names in the dataset labels
import os

class_ids = set()
for root, dirs, files in os.walk(dataset.location):
    for f in files:
        if f.endswith('.txt') and 'labels' in root:
            with open(os.path.join(root, f)) as lf:
                for line in lf:
                    parts = line.strip().split()
                    if parts:
                        class_ids.add(int(parts[0]))
print('Class IDs in labels:', sorted(class_ids))

# Count images
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset.location, split, 'images')
    if os.path.exists(img_dir):
        print(f'{split}: {len(os.listdir(img_dir))} images')

In [ ]:
# Cell 5 — Fine-tune PPE model
# Starts from ppe_model.pt (keeps existing PPE knowledge)
# Low learning rate (lr0=0.001) to preserve existing classes while learning gloves
from ultralytics import YOLO

model = YOLO('/content/ppe_model.pt')   # ← your existing PPE model as base

results = model.train(
    data      = DATA_YAML,
    epochs    = 30,
    imgsz     = 640,
    batch     = 16,
    lr0       = 0.001,       # low LR to avoid forgetting existing PPE classes
    freeze    = 10,          # freeze first 10 layers — only retrain head layers
    project   = '/content/runs/ppe_gloves',
    name      = 'train',
    exist_ok  = True,
    device    = 0,
)
print('Training complete.')

In [ ]:
# Cell 6 — Validate
metrics = model.val()
print('mAP50:    ', round(metrics.box.map50, 4))
print('mAP50-95: ', round(metrics.box.map, 4))

In [ ]:
# Cell 7 — Download the fine-tuned model
from google.colab import files
files.download('/content/runs/ppe_gloves/train/weights/best.pt')

In [ ]:
# Cell 9 — Save to Google Drive (optional but recommended)
# This lets you keep the model after the Colab session ends
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dest = '/content/drive/MyDrive/skyEye_models/gloves_model.pt'
os.makedirs(os.path.dirname(dest), exist_ok=True)
shutil.copy('/content/runs/gloves/train/weights/best.pt', dest)
print('Saved to Google Drive:', dest)

## After downloading `best.pt`

1. Rename it to `gloves_model.pt` and place it in your project's `weights/` folder
2. Update `config.yaml`:
   ```yaml
   model_path: weights/gloves_model.pt
   ```
3. Update `data/class_names.yaml` — set the class names to match what you trained on:
   ```yaml
   names:
     0: Gloves
     1: NO-Gloves
   violation_classes:
     - NO-Gloves
   safe_classes:
     - Gloves
   ```
4. Restart the app and test with `test_detection.py`